<a href="https://colab.research.google.com/github/MahdiTheGreat/Structured-machine-learning/blob/main/project%203/sml_project_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# setups for Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os
import sys
import subprocess
import time

# FIX: Clear Jupyter's import cache for our project.
# If a previous run failed to find the module, Python permanently caches the failure (negative cache).
# for key in list(sys.modules.keys()):
#     if key.startswith("sml_project3"):
#         del sys.modules[key]

# FIX: In Colab/Jupyter, os.chdir() persists across cell executions.
# If we re-run this script, we don't want to descend deeper into the folders.
# We explicitly reset the working directory to the folder containing this script.
# script_dir = os.path.dirname(os.path.abspath(__file__))
# scratch_dir = os.path.abspath(os.path.join(script_dir, "..", "scratch"))

# os.chdir(script_dir)
# if script_dir not in sys.path:
#     sys.path.insert(0, script_dir)
# if scratch_dir not in sys.path:
#     sys.path.insert(0, scratch_dir)

def run_cmd(cmd):
    subprocess.check_call(cmd, shell=True)

installed_anything = False

scratch_dir = ""
scratch_proj3 = os.path.join(scratch_dir, "sml_project3")

if IN_COLAB:
    !git clone https://github.com/olsson-group/sml_project3.git
    !pip install -r sml_project3/requirements.txt

    # Installing torch_scatter and torch_cluster is cumbersome because they are
    # highly optimized implementations that depends on architecure and cuda version.
    # Building these wheels can take up to 20-30 minutes.

    # !pip install torch-scatter -f https://data.pyg.org/whl/torch-$(python3 -c "import torch; print(torch.__version__)")+cu$(python3 -c "import torch; print(torch.version.cuda.split('.')[0]+torch.version.cuda.split('.')[1])")/torch_scatter.html
    # !pip install torch-cluster -f https://data.pyg.org/whl/torch-$(python3 -c "import torch; print(torch.__version__)")+cu$(python3 -c "import torch; print(torch.version.cuda.split('.')[0]+torch.version.cuda.split('.')[1])")/torch_cluster.html

# if not os.path.exists(scratch_proj3):
#     print("Cloning sml_project3 utilities to scratch...")
#     #os.makedirs(scratch_dir, exist_ok=True)
#     run_cmd(f"git clone https://github.com/olsson-group/sml_project3.git {scratch_proj3}")
#     print("Installing requirements...")
#     run_cmd(f"{sys.executable} -m pip install -r {os.path.join(scratch_proj3, 'requirements.txt')}")
#     installed_anything = True

import shutil

# FIX: The cloned repo has an "inception" style nested package (sml_project3/sml_project3)
# which hopelessly confuses Jupyter and Python's import cache.
# We physically move all python files from the inner folder to the outer folder,
# and delete the inner folder, flattening the repository so it matches Project 2 exactly!
# inner_dir = os.path.join(scratch_proj3, "sml_project3")
# if os.path.exists(inner_dir):
#     for f in os.listdir(inner_dir):
#         src = os.path.join(inner_dir, f)
#         dst = os.path.join(scratch_proj3, f)
#         if os.path.exists(dst):
#             if os.path.isdir(dst):
#                 shutil.rmtree(dst)
#             else:
#                 os.remove(dst)
#         shutil.move(src, dst)
#     shutil.rmtree(inner_dir)

try:
    import torch_geometric
    import mdtraj
# except ImportError:
#     print("Installing torch_geometric and mdtraj...")
#     run_cmd(f"{sys.executable} -m pip install torch_geometric mdtraj e3nn")
#     installed_anything = True
except AttributeError as e:
    print("\n" + "="*60)
    print("CRITICAL: PyTorch Geometric circular import detected!")
    print("You MUST restart the Colab Runtime (Runtime -> Restart session) right now.")
    print(f"Error details: {e}")
    print("="*60 + "\n")
    sys.exit(1)

try:
    import torch_scatter
except ImportError:
    print("Installing torch-scatter & torch-cluster (pre-built wheels)...")
    import torch
    torch_version = torch.__version__.split('+')[0]
    cuda_version = "cu" + torch.version.cuda.replace(".", "") if torch.version.cuda else "cpu"
    url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_version}.html"
    run_cmd(f"{sys.executable} -m pip install torch-scatter -f {url}")
    run_cmd(f"{sys.executable} -m pip install torch-cluster -f {url}")
    #installed_anything = True

# if installed_anything:
#     print("\n" + "="*60)
#     print("CRITICAL: New packages were installed!")
#     print("You MUST restart the Colab Runtime (Runtime -> Restart session) before running this script again.")
#     print("This prevents PyTorch circular import errors.")
#     print("="*60 + "\n")
#     sys.exit(1)



fatal: destination path 'sml_project3' already exists and is not an empty directory.


In [42]:
# FIX: painn.py contains a device mismatch bug where equivariant_node_features is created on CPU.
# We patch it dynamically before it gets imported!
painn_file = os.path.join(scratch_proj3, "painn.py")
if os.path.exists(painn_file):
    with open(painn_file, "r") as f:
        painn_code = f.read()
    import re
    painn_code_fixed = re.sub(
        r"(batch\.equivariant_node_features\s*=\s*torch\.zeros\(\s*batch\.pos\.shape\[0\],\s*self\.n_features,\s*3)(\s*\))",
        r"\1, device=batch.pos.device\2",
        painn_code
    )
    if painn_code_fixed != painn_code:
        painn_code = painn_code_fixed

    # Fix 2: Device/shape mismatch for the time tensor 't' during CFM sampling
    target_str_2 = "t_batch = t[batch.batch, None]"
    fixed_str_2 = """t = t.to(batch.pos.device)
        if t.size(0) == batch.pos.size(0):
            t_batch = t.unsqueeze(-1)
        else:
            t_batch = t[batch.batch.to(t.device), None]"""
    if target_str_2 in painn_code:
        painn_code = painn_code.replace(target_str_2, fixed_str_2)

    # Fix 3: The batch metadata (atom_idx, edge_index) might be on CPU during evaluation.
    target_str_3 = "batch = batch.clone()"
    fixed_str_3 = "batch = batch.clone().to(batch.pos.device)"
    if target_str_3 in painn_code:
        painn_code = painn_code.replace(target_str_3, fixed_str_3)

    with open(painn_file, "w") as f:
        f.write(painn_code)
    print("Successfully patched painn.py bugs.")

Successfully patched painn.py bugs.


In [43]:
# import python packages
import os
import numpy as np
import torch
import torch_geometric as geom
# import torch_scatter


import matplotlib.pyplot as plt
from scipy.stats import entropy

# The sml_project3 utilities expect the current working directory to be the repo root
# so they can find relative paths like data/ and oracle/.weights/

# repo_dir = os.path.join(script_dir, "sml_project3")
# os.chdir(repo_dir)

# # import project files
# from sml_project3 import data
# from sml_project3 import mlops
# from sml_project3 import util
# from sml_project3 import model
# from sml_project3 import painn
# from sml_project3 import torsion

# from oracle import util as outil

# # reload
# import importlib
# importlib.reload(painn)
# importlib.reload(util)
# importlib.reload(model)

# device

# settings
if IN_COLAB:
    os.chdir(f"/content/sml_project3")

    # import project files
    from sml_project3.sml_project3 import data
    from sml_project3.sml_project3 import mlops
    from sml_project3.sml_project3 import util
    from sml_project3.sml_project3 import model
    from sml_project3.sml_project3 import painn
    from sml_project3.sml_project3 import torsion

else:
    # import project files
    from sml_project3 import data
    from sml_project3 import mlops
    from sml_project3 import util
    from sml_project3 import model
    from sml_project3 import painn
    from sml_project3 import torsion

from oracle import util as outil

# reload
import importlib
importlib.reload(painn)
importlib.reload(util)
importlib.reload(model)


<module 'sml_project3.sml_project3.model' from '/content/sml_project3/sml_project3/model.py'>

In [3]:
# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [44]:
# MODEL: Basedistribution

class BaseDistribution:
    def __init__(self, device="cpu"):
        # The std of the coordinates is 0.1277
        self.std = 0.1277
        self.device = device

    def sample(self, n_samples):
        # HINT: This function must return a tensor of dimension [n_atoms, 3]
        # Generates a tensor of dimension [n_samples * 15, 3] for a batch of pent-1-ene
        # Sample isotropic normal
        samples = torch.randn(n_samples, 15, 3, device=self.device) * self.std
        # Remove translational symmetry by moving center of mass to origin
        samples = samples - samples.mean(dim=1, keepdim=True)
        return samples.view(-1, 3)

# MODEL: BASELINE MODEL

class BaselineModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Input features: 3 coordinates + 1 time scalar = 4
        self.net = torch.nn.Sequential(
            torch.nn.Linear(4, 64),
            torch.nn.GELU(),
            torch.nn.Linear(64, 64),
            torch.nn.GELU(),
            torch.nn.Linear(64, 3)
        )


    def forward(self, t, batch):
        # HINT: This function must return a tensor of dimension [n_atoms, 3]
        # batch.pos is [n_atoms, 3]
        # Ensure t is on the exact same device as the batch features
        t = t.to(batch.pos.device)

        # During training, t is [batch_size]. During CFM sampling, t is [n_atoms].
        if t.size(0) == batch.pos.size(0):
            t_batch = t
        else:
            # Safely move batch.batch to the same device before indexing
            t_batch = t[batch.batch.to(t.device)]

        t_expanded = t_batch.unsqueeze(1) if t_batch.dim() == 1 else t_batch
        x = torch.cat([batch.pos, t_expanded], dim=1)
        velocity = self.net(x)
        return velocity


# MODEL: Equivariant readout

class EquivariantReadout(torch.nn.Module):
    def __init__(self, n_features):
        super().__init__()
        # MLP mapping invariant features to weights for the equivariant vectors
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(n_features, n_features),
            torch.nn.SiLU(),
            torch.nn.Linear(n_features, n_features)
        )

    def forward(self, batch):
        # TODO: Implement the readout function for the painn model which embeds all nodes
        # with invariant and equivariant features

        # HINT: The input batch will have in and equivariant on it with shapes
        # batch.equivariant_features.shape == (n_atoms, n_features, 3)
        # batch.invariant_features.shape == (n_atoms, n_features)

        # HINT: This function should return a tensor of shape [n_atoms, 3]

        # Use the variable names from painn.py
        inv = batch.invariant_node_features
        equiv = batch.equivariant_node_features

        # Calculate scalar weights from invariant features
        weights = self.mlp(inv) # [n_atoms, n_features]
        weights = weights.unsqueeze(-1) # [n_atoms, n_features, 1]

        # Linear combination of equivariant features
        v_t = (equiv * weights).sum(dim=1) # [n_atoms, 3]
        return v_t

def train_cfm(cfm_model, name, epochs=1000, print_freq=25):
    cfm_model = cfm_model.to(device)
    cfm_model.train()
    print(f"\n--- Training {name} for {epochs} epochs ---")
    timer = util.Timer()
    os.makedirs(f"results/{name}", exist_ok=True)
    step = 0
    for epoch in range(epochs):
        epoch_loss = 0
        for i, batch in enumerate(dataloader):
            batch = batch.to(device)
            # t is a random time between 0 and 1 for each graph in the batch
            t = torch.rand(len(batch)).type_as(batch.pos).to(device)
            loss = cfm_model.get_loss(t, batch)

            if epoch == 0 and i == 0:
                print(f"\nArchitecture for {name} after first forward pass:")
                print(cfm_model)
                print("\n")

            cfm_model.training_step(loss)
            epoch_loss += loss.item()
            step += 1

            if (step + 1) % 1000 == 0 and not DEBUG_MODE:
                mlops.save(cfm_model, f"results/{name}/model_{step}.pkl")

        epoch_loss /= max(1, len(dataloader))
        cfm_model.on_epoch_end(epoch_loss)
        if epoch % print_freq == 0 or epoch == epochs - 1:
            print(f"Epoch: {epoch}, Step: {step}, Time: {timer}, Loss: {epoch_loss:.4f}")

    mlops.save(cfm_model, f"results/{name}/model_latest.pkl")
    return cfm_model

# SAMPLING & EVALUATION:

def generate_and_evaluate(cfm, name):
    print(f"\n=== Evaluating {name} ===")
    cfm.eval()
    n_samples = 10 if DEBUG_MODE else 100
    base_dataset = data.BaseDistributionDataset(n_samples, basedistribution)
    print(f"Loaded BaseDistributionDataset: {len(base_dataset)} molecules")
    base_loader = geom.loader.DataLoader(base_dataset, batch_size=512, shuffle=False)

    print(f"Sampling {n_samples} molecules...")
    samples = cfm.sample(base_loader, n_steps=100)
    samples = samples.cpu().numpy()

    # Evaluate Torsions
    torsion_evaluator = torsion.TorsionEvaluator()
    sampled_torsions = torsion_evaluator.evaluate(samples[-1])

    # Compute reference dataset torsions
    ref_dataset = data.Pentene1Dataset()
    if DEBUG_MODE:
        ref_dataset = torch.utils.data.Subset(ref_dataset, range(10))
    ref_loader = geom.loader.DataLoader(ref_dataset, batch_size=512, shuffle=False)

    ref_pos = []
    for batch in ref_loader:
        ref_pos.append(batch.pos.view(-1, 15, 3).numpy())
    ref_pos = np.concatenate(ref_pos, axis=0)
    ref_torsions = torsion_evaluator.evaluate(ref_pos)

    # Evaluate energies with Oracle
    print("Evaluating energies with Oracle...")
    oracle = outil.load_oracle(device=device)

    # Dataset of generated samples
    sol_samples = torch.tensor(samples[-1], device=device, dtype=torch.float32)
    samples_dataset = data.Pentene1Dataset(sol_samples.cpu().numpy())
    samples_dataloader = geom.loader.DataLoader(samples_dataset, batch_size=512, shuffle=False)

    oracles = []
    for batch in samples_dataloader:
        batch = batch.to(device)
        oracle_energies = oracle.get_energy(batch)
        if hasattr(oracle_energies, "detach"):
            oracle_energies = oracle_energies.detach().cpu().numpy()
        oracles.append(oracle_energies)
    oracle_energies = np.concatenate(oracles, axis=0)

    # Reference energies
    refs = []
    for batch in ref_loader:
        batch = batch.to(device)
        ref_energy = oracle.get_energy(batch)
        if hasattr(ref_energy, "detach"):
            ref_energy = ref_energy.detach().cpu().numpy()
        refs.append(ref_energy)
    ref_energies = np.concatenate(refs, axis=0)

    # KL Divergence Metric
    def kl_div(p, q, bins=50):
        min_val = min(p.min(), q.min())
        max_val = max(p.max(), q.max())
        p_hist, _ = np.histogram(p, bins=bins, range=(min_val, max_val), density=True)
        q_hist, _ = np.histogram(q, bins=bins, range=(min_val, max_val), density=True)
        p_hist = p_hist + 1e-8
        q_hist = q_hist + 1e-8
        return entropy(p_hist, q_hist)

    # KL(ref || generated): standard convention for measuring how well the generated
    # distribution approximates the reference (ground truth) distribution.
    kl_torsion = kl_div(ref_torsions, sampled_torsions)
    kl_energy = kl_div(ref_energies, oracle_energies)

    print(f"KL Divergence (Torsion): {kl_torsion:.4f}")
    print(f"KL Divergence (Energy):  {kl_energy:.4f}")

    # Visualization
    os.makedirs(f"results/{name}", exist_ok=True)

    plt.figure(figsize=(8, 6))
    plt.hist(ref_torsions, bins=50, density=True, alpha=0.5, label='Reference')
    plt.hist(sampled_torsions, bins=50, density=True, alpha=0.5, label='Generated')
    plt.title(f'{name} CFM: Torsion Distribution (KL: {kl_torsion:.4f})')
    plt.xlabel('Torsion Angle')
    plt.ylabel('Density')
    plt.legend()
    plt.savefig(f'results/{name}/torsion_dist.png')
    plt.close()

    plt.figure(figsize=(8, 6))
    plt.hist(ref_energies, bins=50, density=True, alpha=0.5, label='Reference')
    plt.hist(oracle_energies, bins=50, density=True, alpha=0.5, label='Generated')
    plt.title(f'{name} CFM: Energy Distribution (KL: {kl_energy:.4f})')
    plt.xlabel('Energy')
    plt.ylabel('Density')
    plt.legend()
    plt.savefig(f'results/{name}/energy_dist.png')
    plt.close()

In [45]:
import sys
DEBUG_MODE = '--test' in sys.argv
n_epochs = 1 if DEBUG_MODE else 1000
if DEBUG_MODE:

    print("\n[WARNING] FAST DEBUG MODE")

dataset = data.Pentene1Dataset()
print(f"Loaded Pentene1Dataset: {len(dataset)} molecules")
if DEBUG_MODE:
    dataset = torch.utils.data.Subset(dataset, range(10))
    print(f"DEBUG MODE: subset dataset to {len(dataset)} molecules")
dataloader = geom.loader.DataLoader(dataset, batch_size=128, shuffle=True)

readout = EquivariantReadout(n_features=8).to(device)
basedistribution = BaseDistribution(device=device)

baseline_score = BaselineModel().to(device)
baseline_cfm = model.CFM(baseline_score, basedistribution).to(device)

painn_score = painn.Painn(n_features=8, readout=readout).to(device)
painn_cfm = model.CFM(painn_score, basedistribution).to(device)

Loaded Pentene1Dataset: 22000 molecules


In [19]:
baseline_cfm = train_cfm(baseline_cfm, "Baseline", epochs=100,print_freq=10)



--- Training Baseline for 100 epochs ---

Architecture for Baseline after first forward pass:
CFM(
  (score): BaselineModel(
    (net): Sequential(
      (0): Linear(in_features=4, out_features=64, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): GELU(approximate='none')
      (4): Linear(in_features=64, out_features=3, bias=True)
    )
  )
)


Epoch: 0, Step: 172, Time: 0:00:01, Loss: 0.0956
Epoch: 10, Step: 1892, Time: 0:00:18, Loss: 0.0745
Epoch: 20, Step: 3612, Time: 0:00:34, Loss: 0.0743
Epoch: 30, Step: 5332, Time: 0:00:51, Loss: 0.0742
Epoch: 40, Step: 7052, Time: 0:01:08, Loss: 0.0742
Epoch: 50, Step: 8772, Time: 0:01:25, Loss: 0.0744
Epoch: 60, Step: 10492, Time: 0:01:42, Loss: 0.0744
Epoch: 70, Step: 12212, Time: 0:01:58, Loss: 0.0744
Epoch: 80, Step: 13932, Time: 0:02:15, Loss: 0.0742
Epoch: 90, Step: 15652, Time: 0:02:32, Loss: 0.0742
Epoch: 99, Step: 17200, Time: 0:02:47, Loss: 0.0745


In [38]:
generate_and_evaluate(baseline_cfm, "Baseline")


=== Evaluating Baseline ===
Loaded BaseDistributionDataset: 100 molecules
Sampling 100 molecules...


KeyboardInterrupt: 

In [46]:
painn_cfm = train_cfm(painn_cfm, "PaiNN", epochs=1)


--- Training PaiNN for 1 epochs ---

Architecture for PaiNN after first forward pass:
CFM(
  (score): Painn(
    (embedding): Embedding(15, 7)
    (layers): Sequential(
      (0): Message(
        (positional_encoder): PositionalEncoder()
        (phi): MLP(
          (mlp): Sequential(
            (0): Linear(in_features=8, out_features=8, bias=True)
            (1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
            (2): SiLU()
            (3): Linear(in_features=8, out_features=8, bias=True)
            (4): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
            (5): SiLU()
            (6): Linear(in_features=8, out_features=24, bias=True)
          )
        )
        (w): MLP(
          (mlp): Sequential(
            (0): Linear(in_features=8, out_features=8, bias=True)
            (1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
            (2): SiLU()
            (3): Linear(in_features=8, out_features=8, bias=True)
            (4): LayerNorm((8,),

In [47]:
generate_and_evaluate(painn_cfm, "PaiNN")


=== Evaluating PaiNN ===
Loaded BaseDistributionDataset: 100 molecules
Sampling 100 molecules...
Evaluating energies with Oracle...
KL Divergence (Torsion): 3.4258
KL Divergence (Energy):  13.9521


# Project 3 Report Answers & Discussion

## Question 1: Boltzmann Distribution Invariance
**Q:** *The potential u(x) of 'pent-1-ene' is E(3) invariant. What does that imply for the corresponding Boltzmann distribution?*
**Answer:** The Boltzmann distribution is defined as $p(x) = \frac{1}{Z} \exp(-\beta u(x))$. Because the potential energy $u(x)$ is invariant to E(3) transformations (translations and rotations), applying any rotation or translation $R$ to the coordinates does not change the energy: $u(Rx) = u(x)$. Consequently, the probability density remains the same: $p(Rx) = p(x)$. This means the Boltzmann distribution itself is E(3) invariant.

## Question 2: Velocity Field Equivariance vs Target Invariance
**Q:** *Why does our model $v_\theta$ need to be equivariant when our target is invariant?*
**Answer:** The target distribution $p(x)$ is a scalar probability density (which is invariant). However, the model $v_\theta(t, x_t)$ predicts a **velocity field**, which is a collection of vectors detailing how each atom should move in 3D space to transform noise into a molecular conformation. Vectors are inherently directional. If we rotate the entire molecule, the directions the atoms need to travel must rotate by the exact same amount. Thus, the velocity field must be **equivariant** to E(3) transformations, ensuring the generative trajectory is valid regardless of the molecule's spatial orientation.

## Question 3: Importance Sampling Computation
**Q:** *When you have an exact likelihood method, what would you need to compute importance weights $w_i = \exp(-\beta u(x_i)) / p_\theta(x_i)$?*
**Answer:** To compute these weights, we would need to calculate the exact log-likelihood $\log p_\theta(x_i)$ of each generated sample under our learned model. For a continuous normalizing flow (like CFM), computing this requires evaluating the instantaneous change-of-variables formula by integrating the trace of the Jacobian of the velocity field over time. This trace computation is extremely expensive (often requiring Hutchinson's trace estimator) and is bypassed in simulation-free CFM training, which is why we don't compute likelihoods natively here.

## Question 4: Effective Sample Size
**Q:** *What ballpark range would the effective sample size $n_{eff} = (\sum w_i)^2 / \sum (w_i^2)$ be? Compare baseline vs PaiNN.*
**Answer:** The effective sample size measures how closely the generated distribution matches the true Boltzmann distribution. A perfectly matched generative model would have $n_{eff} \approx N$ (where $N$ is the number of samples). In practice, because molecular conformations are highly peaked and complex, the baseline model (which lacks equivariance and structural priors) will propose many invalid or high-energy structures, leading to a few samples dominating the weights and dropping $n_{eff}$ close to 0. The PaiNN model, by leveraging E(3) equivariance and message passing, stays much closer to the true energy manifold. Therefore, PaiNN will have a significantly higher $n_{eff}$ compared to the baseline, indicating much higher sample efficiency.
